---
---
# Universidad Federico Santa María - 2026

<img src="https://fdiaz1968.github.io/Finance-MBA/images/logo_utfsm.png" alt="Universidad Técnica Federico Santa María - Departamento de Ingeniería Comercial" style="width: 480px !important; max-width: 100% !important; height: auto !important;"/>

## FINANZAS

### Profesor Fernando Díaz H.
---

# 🏭 El Modelo de Tres Factores de Fama-French

El **CAPM** explica el retorno esperado de una acción con un solo factor: su beta de mercado. Sin embargo, desde fines de los años 80 la evidencia empírica mostró que el CAPM deja retornos sistemáticamente sin explicar: las acciones de empresas **pequeñas** y las acciones **"valor"** (con un alto ratio *book-to-market*) tienden a rendir, en promedio, más de lo que su beta de mercado predice. Es decir, sus **alfas de Jensen** son significativamente positivos.

Fama y French (1993) propusieron explicar esos retornos "anómalos" agregando dos factores adicionales al CAPM:

$$R_{i,t}=\alpha_{i}+\beta_{i}\,MRP_{t}+s_{i}\,SMB_{t}+h_{i}\,HML_{t}+\varepsilon_{i,t}$$

donde $R_{i,t}=r_{i,t}-r_{f,t}$ es el retorno **en exceso** de la acción, y:

* $MRP_{t}=r_{M,t}-r_{f,t}$ es la prima de mercado (el mismo factor del CAPM, también llamado `Mkt-RF`);
* $SMB_{t}$ (*Small Minus Big*) es la prima por **tamaño**: el retorno de un portafolio de acciones pequeñas menos uno de acciones grandes;
* $HML_{t}$ (*High Minus Low*) es la prima por **valor**: el retorno de un portafolio de acciones "valor" (alto *book-to-market*) menos uno de acciones "crecimiento" (bajo *book-to-market*);
* $s_{i}$ y $h_{i}$ son las sensibilidades (*loadings*) de la acción a cada factor.

Como la regresión ya está escrita en retornos **en exceso**, el intercepto $\alpha_{i}$ es, igual que en el notebook de estimación de betas, el **alfa de Jensen**: el retorno que el modelo no logra explicar. La diferencia es que ahora "el modelo" no es solo el mercado, sino los tres factores en conjunto.

En este notebook:

1. Descargaremos los factores de Fama-French directamente desde el sitio de **Kenneth French** (Dartmouth).
2. Descargaremos precios de cinco acciones y calcularemos sus retornos mensuales en exceso.
3. Estimaremos, para cada acción, el **modelo de mercado (CAPM)** y el **modelo de tres factores**, ambos con retornos en exceso.
4. Compararemos el alfa de Jensen del CAPM con el alfa del modelo de tres factores, para ver si SMB y HML explican parte de lo que el CAPM dejaba como "anómalo".

> 💡 **Idea central:** si una acción tiene un alfa de Jensen positivo en el CAPM simplemente porque es una acción pequeña o "valor" —no porque el mercado la esté "sub-valorando"—, entonces ese alfa debería **reducirse o desaparecer** al controlar por SMB y HML.

## 📦 Cargando las librerías

* **`yfinance`**: descarga de precios de acciones.
* **`pandas_datareader`**: descarga los factores de Fama-French directamente desde el sitio de Kenneth French, sin tener que manipular el archivo `.zip` a mano.
* **`pandas`** y **`numpy`**: transformación de datos y cálculos numéricos.
* **`matplotlib`**: gráficos.
* **`statsmodels`**: regresiones por mínimos cuadrados ordinarios (MCO) y tablas de resultados.

In [ ]:
%%capture
%pip install yfinance pandas_datareader statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import pandas_datareader.data as web
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from IPython.display import display, Markdown

---
## 📥 Descargando los factores de Fama-French

`pandas_datareader` sabe leer directamente el archivo que publica Kenneth French, sin que debamos descargar y descomprimir el `.zip` nosotros mismos. Le pedimos el dataset `'F-F_Research_Data_Factors'`, en frecuencia mensual; el resultado es una lista de dos tablas (mensual y anual), por lo que nos quedamos con la primera, `[0]`.

Los factores vienen expresados en **porcentaje**, por lo que los dividimos por 100. También renombramos `Mkt-RF` como `MRP` (*Market Risk Premium*), para que sea consistente con la notación del notebook de estimación de betas.

In [ ]:
ff_raw = web.DataReader('F-F_Research_Data_Factors', 'famafrench',
                        start='2020-12-01', end='2025-12-31')[0]

ff_factors = (ff_raw / 100).reset_index()
ff_factors = ff_factors.rename(columns={'Mkt-RF': 'MRP', 'Date': 'date'})
ff_factors['date'] = ff_factors['date'].dt.to_timestamp()
ff_factors = ff_factors[['date', 'MRP', 'SMB', 'HML', 'RF']]

print(f"Observaciones mensuales: {len(ff_factors)} "
      f"(de {ff_factors['date'].min().date()} a {ff_factors['date'].max().date()})")
ff_factors.head()

## 📥 Descargando los precios de las acciones

Trabajaremos con cinco acciones de sectores distintos:

* **Apple Inc. (`AAPL`)**: tecnología.
* **Walmart Inc. (`WMT`)**: comercio minorista.
* **Intel Corporation (`INTC`)**: semiconductores.
* **Exxon Mobil Corporation (`XOM`)**: energía.
* **Lockheed Martin Corporation (`LMT`)**: aeroespacial y defensa.

Al igual que en el notebook de estimación de betas, partimos en **diciembre de 2020** para disponer del precio base de enero de 2021, y trabajamos con retornos logarítmicos **mensuales**.

In [ ]:
tick = ['AAPL', 'WMT', 'INTC', 'XOM', 'LMT']

price_data = yf.download(tick, start='2020-12-01', end='2026-01-01', progress=False)['Close']
price_data = price_data[tick]

precios_mensuales = price_data.resample('ME').last()
log_ret = np.log(precios_mensuales / precios_mensuales.shift(1)).dropna()

# Normalizamos el índice al primer día de cada mes, para calzar con los factores de Fama-French
log_ret.index = log_ret.index.to_period('M').to_timestamp()
log_ret.index.name = 'date'
log_ret = log_ret.reset_index()

log_ret.head()

---
## 🔗 Combinando retornos y factores

Unimos los retornos de las acciones con los factores de Fama-French por fecha, y calculamos el retorno **en exceso** de cada acción restando `RF` (la tasa libre de riesgo mensual que reporta el propio Kenneth French).

> 💡 A diferencia del notebook de estimación de betas, aquí **no** necesitamos calcular nosotros mismos la tasa libre de riesgo ni el retorno del mercado: ambos vienen incluidos en el archivo de factores (`RF` y `MRP`, respectivamente), construidos y validados por Kenneth French a partir de todo el universo de acciones de EE.UU. `SMB` y `HML` se construyen ordenando, cada junio, todas las acciones de EE.UU. por tamaño y por *book-to-market*, formando 6 portafolios de referencia (small/big x low/medium/high) con los puntos de corte de NYSE; `SMB` es el retorno promedio de los portafolios pequeños menos los grandes, y `HML` el de los portafolios "valor" menos los "crecimiento". No replicamos ese procedimiento aquí: lo usamos ya construido.

In [ ]:
db = log_ret.merge(ff_factors, on='date', how='inner')
db[tick] = db[tick].sub(db['RF'], axis=0)

print(f"Observaciones combinadas: {len(db)}")
db.head()

## 📊 Estadística descriptiva de los factores

Antes de estimar, veamos el comportamiento de los tres factores durante la muestra: su promedio y volatilidad mensual y anualizada.

In [ ]:
desc_factores = db[['MRP', 'SMB', 'HML']].agg(['mean', 'std']).T
desc_factores.columns = ['Promedio mensual', 'Desv. est. mensual']
desc_factores['Promedio anualizado'] = desc_factores['Promedio mensual'] * 12
desc_factores['Desv. est. anualizada'] = desc_factores['Desv. est. mensual'] * np.sqrt(12)

desc_factores.style.format("{:.2%}")

### 📈 Evolución acumulada de los factores

Grafiquemos el crecimiento de $1 invertido en cada factor a lo largo de la muestra ($e^{\sum r_{t}}$, ya que trabajamos con retornos logarítmicos). Es habitual que `MRP` muestre una tendencia positiva y volátil, mientras que `SMB` y `HML` fluctúan mucho más cerca de cero.

> ⚠️ Un `SMB` o `HML` con tendencia negativa en un período dado no invalida el modelo: son primas de riesgo **esperadas** en promedio a largo plazo, no retornos garantizados en cualquier ventana de cinco años.

In [ ]:
crecimiento = np.exp(db.set_index('date')[['MRP', 'SMB', 'HML']].cumsum())

fig, ax = plt.subplots(figsize=(10, 6))
for col in crecimiento.columns:
    ax.plot(crecimiento.index, crecimiento[col], linewidth=2, label=col)

ax.axhline(1, color='gray', linestyle=':', linewidth=1)
ax.set_ylabel("Crecimiento de $1 invertido")
ax.set_title("Evolución acumulada de los factores de Fama-French")
ax.legend()
plt.tight_layout()
plt.show()

---
## 🧮 Modelo de Mercado (CAPM) con retornos en exceso

Primero, para cada acción, estimamos el modelo de mercado clásico —solo `MRP` como regresor—, igual que en el notebook de estimación de betas. Como ya trabajamos con retornos en exceso, el intercepto $\hat{\alpha}$ es el **alfa de Jensen**.

In [ ]:
modelos_capm = {a: sm.OLS(db[a], sm.add_constant(db['MRP'])).fit() for a in tick}

filas = []
for a, m in modelos_capm.items():
    filas.append({'Acción': a, 'Alfa mensual': m.params['const'], 'Alfa anualizado': m.params['const'] * 12,
                  'Valor-p (alfa)': m.pvalues['const'], 'Beta (MRP)': m.params['MRP'], 'R²': m.rsquared})

tabla_capm = pd.DataFrame(filas).set_index('Acción')
tabla_capm.style.format({'Alfa mensual': '{:.4f}', 'Alfa anualizado': '{:.2%}', 'Valor-p (alfa)': '{:.3f}',
                         'Beta (MRP)': '{:.3f}', 'R²': '{:.3f}'})

In [ ]:
# Tabla de resultados en formato "una columna por acción" (equivalente a stargazer)
tabla_resumen_capm = summary_col(list(modelos_capm.values()), stars=True, model_names=tick,
                                 regressor_order=['const', 'MRP'],
                                 info_dict={'N': lambda m: f"{int(m.nobs)}", 'R²': lambda m: f"{m.rsquared:.3f}"})
print(tabla_resumen_capm)

---
## 🧮 Modelo de Tres Factores (Fama-French)

Ahora agregamos `SMB` y `HML` como regresores adicionales. El intercepto de esta regresión —que llamaremos $\alpha^{FF3}$— es también un alfa de Jensen, pero **ajustado por tamaño y valor**: mide el retorno que ni el mercado, ni el tamaño, ni el valor de la acción logran explicar.

In [ ]:
X3 = sm.add_constant(db[['MRP', 'SMB', 'HML']])
modelos_ff3 = {a: sm.OLS(db[a], X3).fit() for a in tick}

filas = []
for a, m in modelos_ff3.items():
    filas.append({'Acción': a, 'Alfa FF3 mensual': m.params['const'], 'Alfa FF3 anualizado': m.params['const'] * 12,
                  'Valor-p (alfa)': m.pvalues['const'], 'Beta (MRP)': m.params['MRP'],
                  'Beta (SMB)': m.params['SMB'], 'Beta (HML)': m.params['HML'], 'R²': m.rsquared})

tabla_ff3 = pd.DataFrame(filas).set_index('Acción')
tabla_ff3.style.format({'Alfa FF3 mensual': '{:.4f}', 'Alfa FF3 anualizado': '{:.2%}', 'Valor-p (alfa)': '{:.3f}',
                        'Beta (MRP)': '{:.3f}', 'Beta (SMB)': '{:.3f}', 'Beta (HML)': '{:.3f}', 'R²': '{:.3f}'})

In [ ]:
tabla_resumen_ff3 = summary_col(list(modelos_ff3.values()), stars=True, model_names=tick,
                                regressor_order=['const', 'MRP', 'SMB', 'HML'],
                                info_dict={'N': lambda m: f"{int(m.nobs)}", 'R²': lambda m: f"{m.rsquared:.3f}"})
print(tabla_resumen_ff3)

### 📖 ¿Cómo leer los *loadings* $s_{i}$ y $h_{i}$?

* $s_{i}>0$: la acción se mueve como una acción **pequeña** (junto con `SMB`); $s_{i}<0$, como una acción **grande**.
* $h_{i}>0$: la acción se comporta como una acción **"valor"** (junto con `HML`); $h_{i}<0$, como una acción **"crecimiento"** —empresas con altas expectativas de crecimiento futuro y un *book-to-market* bajo, típicamente tecnológicas.

> ⚠️ Con solo 60 observaciones y tres regresores correlacionados entre sí, los errores estándar de estos *loadings* suelen ser amplios. Interprete el **signo** con más confianza que la magnitud exacta.

### 📄 Exportando las tablas a LaTeX (para las diapositivas)

Las tablas de esta seccion alimentan directamente las diapositivas del curso ("Regresiones de F&F", "Regresiones Market Model"). Como el periodo de la muestra puede cambiar de un semestre a otro, generamos el codigo LaTeX directamente desde los datos actuales, en vez de copiarlo a mano: asi el titulo y el numero de observaciones de la tabla siempre reflejan la muestra que efectivamente se uso.

> 💡 Copie el contenido de `Market_Model.tex` y `Fama_French.tex` directamente dentro de los bloques `\\begin{table}...\\end{table}` de la presentacion, reemplazando la tabla anterior.

In [ ]:
periodo = f"{db['date'].min():%Y-%m} a {db['date'].max():%Y-%m}"

with open("Market_Model.tex", "w") as f:
    f.write(tabla_resumen_capm.as_latex())

with open("Fama_French.tex", "w") as f:
    f.write(tabla_resumen_ff3.as_latex())

print(f"Periodo de la muestra: {periodo}")
print("Tablas exportadas: Market_Model.tex, Fama_French.tex")

---
## 🔍 Comparando el alfa del CAPM con el alfa del modelo de tres factores

Si una acción tenía un alfa de Jensen positivo en el CAPM **porque** es una acción pequeña o "valor" —y no porque el mercado la esté sub-valorando—, entonces su alfa debería **reducirse** al pasar al modelo de tres factores, que ya controla por esas dos características.

In [ ]:
comparacion_alfas = pd.DataFrame({
    'Alfa CAPM (anual)': tabla_capm['Alfa anualizado'],
    'Alfa FF3 (anual)': tabla_ff3['Alfa FF3 anualizado'],
    'R² CAPM': tabla_capm['R²'],
    'R² FF3': tabla_ff3['R²'],
})
comparacion_alfas['Reducción del alfa'] = comparacion_alfas['Alfa CAPM (anual)'] - comparacion_alfas['Alfa FF3 (anual)']

comparacion_alfas.style.format({'Alfa CAPM (anual)': '{:.2%}', 'Alfa FF3 (anual)': '{:.2%}',
                                'R² CAPM': '{:.3f}', 'R² FF3': '{:.3f}', 'Reducción del alfa': '{:.2%}'})

In [ ]:
x = np.arange(len(tick))
ancho = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - ancho/2, comparacion_alfas['Alfa CAPM (anual)'], ancho, label='Alfa CAPM', color='steelblue', edgecolor='black')
ax.bar(x + ancho/2, comparacion_alfas['Alfa FF3 (anual)'], ancho, label='Alfa Fama-French (3F)', color='#d62728', edgecolor='black')

ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(tick)
ax.yaxis.set_major_formatter(lambda v, _: f"{v*100:.0f}%")
ax.set_ylabel("Alfa anualizado")
ax.set_title("Alfa de Jensen: CAPM vs. Modelo de Tres Factores")
ax.legend()
plt.tight_layout()
plt.show()

> 💡 **Cómo leer este gráfico:** si la barra roja (FF3) es más chica que la azul (CAPM) y ambas tienen el mismo signo, parte del alfa "CAPM" se explica por la exposición de la acción a `SMB` y `HML`. Si el alfa FF3 sigue siendo grande —y estadísticamente distinto de cero—, el modelo de tres factores tampoco logra explicar el retorno de esa acción.

---
## ⚠️ Precauciones metodológicas

* **El factor de mercado no es el mismo que en el CAPM del notebook anterior.** Aquí `MRP` es el `Mkt-RF` que publica Kenneth French —construido con (casi) todas las acciones que cotizan en EE.UU.—, mientras que en el notebook de estimación de betas usamos el S&P 500 como proxy. Ambos son razonables, pero no son idénticos.
* **Multicolinealidad.** `MRP`, `SMB` y `HML` no son independientes entre sí, lo que puede inflar los errores estándar de los *loadings* individuales, aun cuando el modelo en su conjunto ajuste bien.
* **Muestra pequeña.** Con 60 observaciones mensuales y tres regresores, hay pocos grados de libertad; los alfas y *loadings* estimados son ruidosos.
* **El modelo de tres factores tampoco es "la verdad".** Es una mejora empírica sobre el CAPM, no una teoría de equilibrio derivada de primeros principios como el CAPM. Existen extensiones con más factores (momentum, calidad, rentabilidad, inversión) que explican aún mejor los retornos.

### 🧭 Conclusión

Al escribir el modelo en retornos **en exceso**, el intercepto de cualquiera de estas regresiones —CAPM o Fama-French— es un alfa de Jensen: el retorno que el modelo, con los factores que incluye, no logra explicar.

El modelo de tres factores agrega `SMB` y `HML` al CAPM. Si el alfa de una acción se reduce al pasar del CAPM al modelo de tres factores, parte de lo que parecía un retorno "anormal" era, en realidad, una prima por tamaño o por valor que el CAPM, con un solo factor, no podía capturar.